## 05b — EDA Appendix: Ablation and Visual Supplements

Complements the main EDA notebook with three additional analyses run after the train/test split was fixed.

**Input:** `workspace.gold.ml_features_train` / `workspace.gold.ml_features_test`  
**Output:** `/tmp/boxplot_value_by_target.png` — also overwrites the saved model artifact

1. Feature ablation: XGBoost trained on value only, categoricals only, and all features, to quantify each group's independent contribution (categoricals alone reach AUC 0.771; value alone reaches 0.530).  
2. Boxplot of log award value by target class, for the report.  
3. Saves the correct final model — XGBoost on categoricals only — overwriting the all-features version from the model notebook.

In [0]:
%pip install xgboost

In [0]:

# ML Analysis — Ablation, EDA supplements, model finalization
 
#  Complements notebook 10. Three goals:
#  1. Feature ablation study (value only vs categoricals only vs all features)
#  2. EDA supplements: boxplot of log_award_value by target, Spearman correlation
#  3. Overwrite the saved model with the correct final version (XGBoost, no award value)

#  ----------

#  %pip install xgboost

#  ----------

# Goal: load fixed train/test tables and define feature lists.
# These are the same tables persisted by notebook 09 — reading from Delta
# guarantees the same split used during training.
import pandas as pd
import numpy as np
from xgboost import XGBClassifier
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import roc_auc_score
import matplotlib.pyplot as plt
from scipy import stats

CAT_FEATURES = ["country_clean", "cpv_division_clean", "procedure_clean", "buyer_type_clean"]
NUM_FEATURES  = ["log_award_value", "value_missing"]
TARGET        = "is_single_bidder"

train_pd = spark.table("workspace.gold.ml_features_train").toPandas()
test_pd  = spark.table("workspace.gold.ml_features_test").toPandas()

X_train = train_pd[CAT_FEATURES + NUM_FEATURES]
y_train = train_pd[TARGET]
X_test  = test_pd[CAT_FEATURES + NUM_FEATURES]
y_test  = test_pd[TARGET]

print("Train:", train_pd.shape, "| Test:", test_pd.shape)

In [0]:
# Conclusion from previous: train/test tables loaded correctly (20,361 / 5,281 rows).
# Goal: ablation study — train XGBoost with (1) value only, (2) categoricals only,
# (3) all features. Quantifies how much signal each feature group contributes independently.

# (1) Value only
prep_val = ColumnTransformer(transformers=[
    ("num", StandardScaler(), NUM_FEATURES),
])
xgb_val_only = Pipeline([
    ("prep", prep_val),
    ("clf", XGBClassifier(n_estimators=200, max_depth=6, learning_rate=0.1,
                          random_state=42, eval_metric="logloss", verbosity=0))
])

# (2) Categoricals only — este es el modelo final
prep_cat = ColumnTransformer(transformers=[
    ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), CAT_FEATURES),
])
xgb_cat_only = Pipeline([
    ("prep", prep_cat),
    ("clf", XGBClassifier(n_estimators=200, max_depth=6, learning_rate=0.1,
                          random_state=42, eval_metric="logloss", verbosity=0))
])

# (3) All features
prep_all = ColumnTransformer(transformers=[
    ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), CAT_FEATURES),
    ("num", StandardScaler(), NUM_FEATURES),
])
xgb_all = Pipeline([
    ("prep", prep_all),
    ("clf", XGBClassifier(n_estimators=200, max_depth=6, learning_rate=0.1,
                          random_state=42, eval_metric="logloss", verbosity=0))
])

print("Training...")
xgb_val_only.fit(train_pd[NUM_FEATURES], y_train)
xgb_cat_only.fit(train_pd[CAT_FEATURES], y_train)
xgb_all.fit(X_train, y_train)

auc_val  = roc_auc_score(y_test, xgb_val_only.predict_proba(test_pd[NUM_FEATURES])[:,1])
auc_cat  = roc_auc_score(y_test, xgb_cat_only.predict_proba(test_pd[CAT_FEATURES])[:,1])
auc_all  = roc_auc_score(y_test, xgb_all.predict_proba(X_test)[:,1])

print("\nFeature ablation study — XGBoost")
print(f"{'Features':<25} {'AUC':>8}")
print("-" * 35)
print(f"{'Value only':<25} {auc_val:>8.4f}")
print(f"{'Categoricals only':<25} {auc_cat:>8.4f}")
print(f"{'All features':<25} {auc_all:>8.4f}")

In [0]:
# Conclusion from previous: ablation confirms categoricals alone explain almost all
# predictive power (AUC 0.771). Value only reaches 0.530 — barely above random.
# Adding value gives 0.007 marginal gain at the cost of CN/CAN parity problems.
# Goal: boxplot of log_award_value by is_single_bidder to visualize the distribution
# difference between competitive and single-bidder contracts.

fig, ax = plt.subplots(figsize=(7, 5))
fig.patch.set_facecolor("#F8F9FA")
ax.set_facecolor("#F8F9FA")
ax.spines[["top", "right"]].set_visible(False)

groups = [
    train_pd.loc[train_pd[TARGET] == 0, "log_award_value"].dropna(),
    train_pd.loc[train_pd[TARGET] == 1, "log_award_value"].dropna(),
]

bp = ax.boxplot(groups, labels=["Competitive (0)", "Single-bidder (1)"],
                patch_artist=True, widths=0.5)

bp["boxes"][0].set_facecolor("#1A6EBD")
bp["boxes"][1].set_facecolor("#E87722")
for element in ["whiskers", "caps", "medians", "fliers"]:
    for item in bp[element]:
        item.set_color("#333333")

ax.set_ylabel("log(award_value_eur)", fontsize=11)
ax.set_title("Contract value distribution by outcome", fontsize=13, fontweight="bold")

plt.tight_layout()
plt.savefig("/tmp/boxplot_value_by_target.png", dpi=150, bbox_inches="tight")
plt.show()